# Lesson 6: Climate futures, from "what happened" to "what are the odds next year"

In Lesson 5 you found a number you didn't trust: **162 frost-free days** at Elora. Today we:

1. **Check NASA POWER temperature against the Elora weather station**, and correct it if needed. This is the same idea as rainfall in Lesson 2, but with one important difference you'll see in Step 4.
2. Turn 30 years of daily weather into **indicators that matter for a rotation**: frost dates, heat during corn pollination, heavy rain on bare soil, summer dry spells, and fall warmth for cover crops.
3. Build **possible next seasons** from those 30 years, and express each risk as **"X in 100"**.

**Job skills in this lesson** (data scientist and ML engineer interview topics):
- *Statistics:* bias vs noise, held-out validation, the **bootstrap**, confidence intervals, **Monte Carlo error**, and deciding rules *before* seeing results
- *Software engineering:* your first **pull request** (Step 0), and keeping a data contract backward compatible
- *Product:* communicating risk as counts ("23 in 100"), not percentages or p-values

At the end there's an **interview practice** section. Answer those questions out loud before you open the model answers.

**Time:** about 2 hours. The station download takes 1–2 minutes the first time.


## Step 0: Review and merge the pull request (on GitHub, before running anything)

Until now, code went straight onto `main`, the branch everyone uses. In a company, nobody does that. Instead:

1. You make a **branch**, a parallel copy of the code where you can work without breaking `main`. I put this lesson's code on a branch called **`lesson-6`**.
2. You open a **pull request (PR)**: "here are my changes, please review them before they join `main`."
3. A colleague **reviews** the PR: reads the changes (the *diff*), asks questions, approves.
4. The PR is **merged** into `main`.

Today **you are the reviewer**. Go to https://github.com/SetarehAlamdar/FieldSense/pulls and open the Lesson 6 pull request. (If you don't see one, open https://github.com/SetarehAlamdar/FieldSense/compare/main...lesson-6 and click **Create pull request**.)

Then:
- Click the **Files changed** tab. Green lines were added and red lines were removed.
- Look at `src/fieldsense/climate.py`. Read the docstring at the top: it explains the three ideas of this lesson.
- Look at `tests/test_climate.py`. Can you tell what each test checks just from its name?
- Look at `profile.py`. Notice the schema went from `0.1` to `0.2`, and that a field I had renamed was kept under its old name, `growing_degree_days_may_sep`. Renaming a field breaks every program that reads it. That's a *breaking change*, and we avoid them.
- If something looks unclear, that's a real review comment. Write it on the line (click the **+** next to it), and ask me here too.
- When you're satisfied, click **Merge pull request** → **Confirm merge**.

**Why this matters for interviews:** "Walk me through how you review code" and "how do you avoid breaking other teams' code?" are common questions. You'll now have a real answer.


## Project setup: run this first, every session

Everything we save, **data files and code modules**, goes into your **Google Drive** under `MyDrive/fieldsense/`. Colab wipes its own disk whenever a session ends, but Drive keeps everything, so each lesson can pick up the previous lessons' files.

Colab will ask permission to access your Drive the first time each session; click **Allow**.


In [ ]:
# ---- Project setup: run this first, every session ----
import os, sys, glob, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/fieldsense"
except ImportError:                       # running outside Colab: use a local folder instead
    PROJECT_DIR = os.path.abspath("fieldsense")
CACHE_DIR = os.path.join(PROJECT_DIR, "data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)                     # %%writefile now saves code modules into Drive too
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)       # lets `import fieldsense_data` find them

# One-time rescue: copy anything still sitting in Colab's temporary folder into Drive
for f in glob.glob("/content/data_cache/*"):
    if not os.path.exists(os.path.join(CACHE_DIR, os.path.basename(f))):
        shutil.copy(f, CACHE_DIR)
        print("Rescued into Drive:", os.path.basename(f))

print("Project folder:", PROJECT_DIR)
print("Saved data files:", sorted(os.listdir(CACHE_DIR)) or "none yet")

PRIVATE_DIR = os.path.join(PROJECT_DIR, "private")       # lab records: NEVER goes to GitHub
PROFILE_DIR = os.path.join(PROJECT_DIR, "profiles")      # field profiles + evidence registry
DEMO_DIR = os.path.join(PROJECT_DIR, "demo")             # figures for slides and website
for d in (PRIVATE_DIR, PROFILE_DIR, DEMO_DIR):
    os.makedirs(d, exist_ok=True)

## Step 1: Install the latest code and run the tests

Same as Lesson 5. This pulls the merged `main` from GitHub, installs it and runs the tests. You should see **version 0.2.0** and **28 passed** (19 from Lesson 5, 9 new ones for climate).

If you see version **0.1.0**, either the PR isn't merged yet, or this Colab session imported the old version earlier. Merge it, then use **Runtime → Restart session** and run again from the top.


In [ ]:
import subprocess
REPO_URL = "https://github.com/SetarehAlamdar/FieldSense.git"
REPO_DIR = os.environ.get("FIELDSENSE_REPO", "/content/FieldSense")

def sh(cmd):
    """Run a shell command; print its output; stop the notebook if it fails."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

url = REPO_URL
try:                                           # private repo: use a token from Colab Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                                       # public repo (or not in Colab): no token needed

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone --quiet {url} {REPO_DIR}")
sh(f'pip install -q -e "{REPO_DIR}[dev]"')

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import fieldsense as fs
print("fieldsense version", fs.__version__, "from", os.path.dirname(fs.__file__))
sh(f"git -C {REPO_DIR} log --oneline -5")
sh(f"cd {REPO_DIR} && python -m pytest -q")

## Step 2: Load the weather and download the Elora station's temperatures

- **NASA POWER** (`weather_elora_corrected.csv` from Lesson 2) has `T2M_MIN`, `T2M_MAX` and `T2M` (daily min, max and mean at 2 m), for a ~50 km square.
- **Elora RCS** (Environment Canada station **41983**) is the automatic Reference Climate Station at the Elora Research Station, **the same research station as your plots**. That makes it close to ideal ground truth for temperature.

In Lesson 2 its rain gauge read low in winter (snow blows past automatic gauges). Thermometers don't have that problem, so it's fine for temperature.

We download 2004–2025 (full years since the station opened), one request per year, cached in Drive. Then `good_days` keeps only days in months with at least 90% of days reported. That's the same rule we used for rain, for the same reason: a month with gaps gives a misleading average.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fieldsense.sources import stations

weather = pd.read_csv(os.path.join(CACHE_DIR, "weather_elora_corrected.csv"), index_col=0, parse_dates=True)
config = json.load(open(os.path.join(CACHE_DIR, "fieldsense_config.json")))
print(f"POWER: {weather.index.min():%Y-%m-%d} to {weather.index.max():%Y-%m-%d}")

st = stations.get_eccc_years(41983, 2004, 2025, cache_dir=CACHE_DIR)
st_tmin, st_tmax, st_tmean = (stations.good_days(st[c]) for c in ("tmin", "tmax", "tmean"))
print(f"Elora RCS: {len(st)} days downloaded; usable days: tmin {len(st_tmin)}, tmax {len(st_tmax)}, tmean {len(st_tmean)}")

## Step 3: First look, where does POWER differ from the station?

Two views:

1. **Average difference by month** (POWER minus station). A positive bar means POWER is too warm that month.
2. **Frost-free days per year**: POWER vs station, side by side.

Before you run it, **make a prediction** (write it down): will POWER's *minimum* temperatures be too warm, too cold, or about right? Why?


In [ ]:
def monthly_diff(model, station):
    both = pd.concat([model.rename("m"), station.rename("s")], axis=1).dropna()
    return (both.m - both.s).groupby(both.index.month).mean()

diff = pd.DataFrame({"night (min)": monthly_diff(weather["T2M_MIN"], st_tmin),
                     "day (max)": monthly_diff(weather["T2M_MAX"], st_tmax)})
fd_power = fs.frost_dates(weather.loc["2004":"2025", "T2M_MIN"])
fd_station = fs.frost_dates(st_tmin)
both_fd = pd.concat({"POWER": fd_power.frost_free_days, "station": fd_station.frost_free_days}, axis=1).dropna()

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
diff.plot.bar(ax=ax[0], color=["#2c5aa0", "#d98c1f"], rot=0)
ax[0].axhline(0, color="k", lw=0.8); ax[0].set_xticklabels("J F M A M J J A S O N D".split())
ax[0].set_ylabel("POWER - station (degC)"); ax[0].set_title("Average difference by month", loc="left")
both_fd.plot(ax=ax[1], marker="o", color=["#2c5aa0", "#333"])
ax[1].set_ylabel("days"); ax[1].set_title("Frost-free days per year", loc="left")
plt.tight_layout(); plt.show()

print(diff.round(2).T.to_string())
print(f"\nFrost-free days, {len(both_fd)} years with good station data: "
      f"POWER median {both_fd.POWER.median():.0f}, station median {both_fd.station.median():.0f}, "
      f"POWER longer by {(both_fd.POWER - both_fd.station).mean():+.0f} days on average")

**How to read it:**
- If the **night** bars are mostly **positive**, POWER's nights are too warm. The likely reason: the ~50 km square averages over towns, lakes and higher ground. On a clear, still night, cold air pools in low open fields like Elora's, so the field gets colder than the average of the square.
- A small error in minimum temperature makes a **big** error in frost dates. In spring, nights warm by only about 0.1–0.2 °C per day. So nights that are 1 °C too warm can move the last frost by **5–10 days**, at both ends of the season.
- Day temperatures are usually closer, because daytime air mixes well over large areas.

Was your prediction right? If the result surprised you, write down what you expected and why. That gap is where you learn the most.


## Step 4: Correct temperature, and test it honestly

**Additive, not multiplicative.** For rain we *multiplied* by a factor, because being 25% short scales with how much fell, and a dry day must stay dry (0 × factor = 0). Temperature is different:
- If POWER's nights are 1.5 °C too warm, they're 1.5 °C too warm at −10 °C *and* at +10 °C. So we **add** an offset.
- Multiplying would be meaningless. Celsius has an arbitrary zero, so multiplying by 1.1 would turn −10 °C into −11 °C but +10 °C into +11 °C, and leave 0 °C unchanged.

So: **offset[month] = average of (station − POWER)** for that calendar month. We do it separately for min, max and mean.

**Validation, same as Lesson 2:** fit on 2004–2014, test on 2015–2025 (years the offsets never saw). We report two numbers before and after:
- **Bias**: the average error. Correction should bring this near 0.
- **MAE** (mean absolute error): the typical size of a *daily* error. This will barely improve, and that's expected.

**Why MAE barely moves: bias vs noise.** Total error = **bias** (a constant lean, which we can fix) + **noise** (day-to-day misses, like a cold pocket on one particular night, which a monthly offset can't fix). Telling these two apart is one of the most useful ideas in data science, and a very common interview question.

What matters for us is the **frost dates**, because they depend on the bias. So we also check frost-free days on the test years.


In [ ]:
SPLIT = 2014
checks = {}
for name, model_col, station in [("min", "T2M_MIN", st_tmin), ("max", "T2M_MAX", st_tmax), ("mean", "T2M", st_tmean)]:
    checks[name] = fs.held_out_offset_check(station, weather[model_col], SPLIT)
    c = checks[name]
    print(f"{name:4s}  bias {c['bias_raw']:+.2f} -> {c['bias_corrected']:+.2f} degC    "
          f"daily MAE {c['mae_raw']:.2f} -> {c['mae_corrected']:.2f} degC")

# Frost-free days on the TEST years, before and after
off_train = fs.monthly_offsets(st_tmin[st_tmin.index.year <= SPLIT], weather.loc[:str(SPLIT), "T2M_MIN"])
test_power = weather.loc[str(SPLIT + 1):"2025", "T2M_MIN"]
ff_raw = fs.frost_dates(test_power).frost_free_days
ff_cor = fs.frost_dates(fs.apply_monthly_offsets(test_power, off_train)).frost_free_days
ff_st = fd_station.frost_free_days
yrs = ff_st.index.intersection(ff_raw.index)
FF_ERR_RAW, FF_ERR_COR = (ff_raw[yrs] - ff_st[yrs]).mean(), (ff_cor[yrs] - ff_st[yrs]).mean()
print(f"\nFrost-free days on {len(yrs)} test years: error {FF_ERR_RAW:+.0f} days -> {FF_ERR_COR:+.0f} days after correction")

**Decide before you look further:** the correction is worth keeping if (a) the test-year bias drops close to 0 and (b) the frost-free-day error shrinks. If the bias was already near 0, we *don't* correct. Adding a correction you don't need only adds noise. Write your decision in the next cell.


In [ ]:
USE_TEMP_CORRECTION = True    # <- your decision from the numbers above (True / False)

## Step 5: Final correction, saved as a NEW version of the weather file

Like in Lesson 2: test with held-out years first, then build the final offsets from **all** years (2004–2025), because more data gives better offsets.

**A data-versioning habit:** we don't overwrite `weather_elora_corrected.csv`. We save **`weather_elora_v2.csv`** next to it. Lesson 5's profiles were built from the old file, and if we overwrote it we could never reproduce them. In ML engineering this is called **data versioning**. Later in the bootcamp you'll use a proper tool for it (DVC), but the idea is just this: never change data in place.

We also:
- record the decision and its test result in `fieldsense_config.json`
- **upgrade `power_temperature` from Modelled to Validated** in the evidence registry, because now we have a test and a result to show

Finally we keep one number for later: **how far off a whole year's average can still be after correction** (the spread of yearly-mean residuals on the test years). The futures use it as `temp_sd`, the "we're still not perfectly sure" noise.


In [ ]:
from fieldsense import Evidence, EvidenceRegistry

w2 = weather.copy()
offsets = {}
if USE_TEMP_CORRECTION:
    for name, col, station in [("min", "T2M_MIN", st_tmin), ("max", "T2M_MAX", st_tmax), ("mean", "T2M", st_tmean)]:
        offsets[name] = fs.monthly_offsets(station, weather[col])
        w2[col] = fs.apply_monthly_offsets(weather[col], offsets[name])
w2.to_csv(os.path.join(CACHE_DIR, "weather_elora_v2.csv"))

resid = checks["min"]["residuals"]
TEMP_SD = float(resid.groupby(resid.index.year).mean().std())      # year-to-year error left after correction
config["temp_correction"] = {
    "used": USE_TEMP_CORRECTION,
    "method": "additive monthly offsets (station - POWER), separately for min/max/mean",
    "reference_station": "Elora RCS, ECCC station 41983 (same research station as the plots)",
    "fit_years": "2004-2025, months with >=90% days reported",
    "offsets_degC": {k: {int(m): round(float(v), 2) for m, v in o.items()} for k, o in offsets.items()},
    "held_out_2015_2025": {k: {"bias_raw": round(c["bias_raw"], 2), "bias_corrected": round(c["bias_corrected"], 2),
                               "mae_raw": round(c["mae_raw"], 2), "mae_corrected": round(c["mae_corrected"], 2)}
                           for k, c in checks.items()},
    "frost_free_days_error_raw_vs_corrected": [round(float(FF_ERR_RAW), 1), round(float(FF_ERR_COR), 1)],
    "yearly_residual_sd_degC": round(TEMP_SD, 2),
    "weather_file": "weather_elora_v2.csv",
}
json.dump(config, open(os.path.join(CACHE_DIR, "fieldsense_config.json"), "w"), indent=2)

reg = EvidenceRegistry.from_json(os.path.join(PROFILE_DIR, "evidence.json"))
if USE_TEMP_CORRECTION:
    c = checks["min"]
    reg.add(Evidence(
        "power_temperature", "validated",
        claim="Daily air temperature at the field (frost dates, heat units), corrected to the Elora station.",
        source="NASA POWER (T2M, T2M_MIN, T2M_MAX), corrected against ECCC Elora RCS (41983)",
        test="Monthly offsets fitted 2004-2014, tested on 2015-2025",
        result=(f"Night-time bias {c['bias_raw']:+.1f} -> {c['bias_corrected']:+.1f} degC; "
                f"frost-free days error {FF_ERR_RAW:+.0f} -> {FF_ERR_COR:+.0f} days"),
        caveat="Fixes the average lean, not single nights: one unusual cold night can still be missed."))
reg.add(Evidence(
    "assumed_risk_thresholds", "assumed",
    claim="Thresholds that turn weather into risks (e.g. 32 degC at corn pollination, 25 mm heavy rain).",
    source="Agronomic rules of thumb in fieldsense.climate.RISKS",
    caveat="To be checked against OMAFRA guidance; change a threshold and the odds change."))
reg.to_json(os.path.join(PROFILE_DIR, "evidence.json"))
print("Offsets for nights (degC by month):", config["temp_correction"]["offsets_degC"].get("min"))
print(f"Remaining year-to-year error (temp_sd): {TEMP_SD:.2f} degC")
print("Saved weather_elora_v2.csv, updated fieldsense_config.json and evidence.json")

## Step 6: Indicators, what 365 days of weather mean for a rotation

A farmer choosing a rotation doesn't need 365 temperatures. They need answers to questions like these:

| Indicator | Question it answers | Why it matters for rotation |
|---|---|---|
| `last_spring_frost_doy`, `first_fall_frost_doy`, `frost_free_days` | How long is the safe season? | Longer-season corn or soybean; time for a crop after wheat |
| `growing_degree_days_may_sep` | How much heat for crops? (base 10 °C) | Which corn and soybean maturity fits |
| `heat_days_pollination` | Days ≥ 32 °C, 10 Jul – 10 Aug | Heat at corn silking cuts yield; wheat avoids it (harvested in July) |
| `heavy_rain_days_spring` | Days ≥ 25 mm, 15 Apr – 15 Jun | Bare soil before corn/soy is when erosion happens; living cover protects it |
| `longest_dry_spell_summer` | Longest run with < 1 mm, Jun – Aug | Drought stress; deep-rooted crops cope better |
| `fall_gdd5_sep_nov` | Warmth after 1 Sept (base 5 °C) | Can a cover crop establish after harvest? |

Notice how these connect to Lesson 5. The **heavy-rain-on-bare-soil** indicator is exactly where the diverse rotation's **living cover** pays off. That's a story the judges can follow: *"it rains hard in spring in X of 100 years, and in a simple rotation that rain hits bare soil 108 days a year."*

All these definitions live in **one** function, `climate.year_indicators`. The profile code calls that function instead of having its own copy. If there were two copies, someone would eventually fix one and not the other.


In [ ]:
hist_v1 = fs.indicators_by_year(weather.loc["1996":"2025"])
hist = fs.indicators_by_year(w2.loc["1996":"2025"])
show_cols = ["frost_free_days", "growing_degree_days_may_sep", "heat_days_pollination",
             "heavy_rain_days_spring", "longest_dry_spell_summer", "fall_gdd5_sep_nov"]
print("Median of 1996-2025:")
print(pd.DataFrame({"before temp correction": hist_v1[show_cols].median(),
                    "after": hist[show_cols].median()}).round(0).to_string())

fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharex=True)
for ax, col in zip(axes.ravel(), show_cols):
    ax.bar(hist.index, hist[col], color="#5a8f3c")
    ax.set_title(col, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(DEMO_DIR, "climate_indicators_elora.png"), dpi=200); plt.show()

**Check:** is the frost-free number now closer to what you'd expect for Elora? (I guessed 140–150 days in my last message. Compare with what the station itself measured in Step 3, which is the real authority, not my guess.)


## Step 7: Is it getting warmer? A trend with a confidence interval

If the 1990s were cooler than today, sampling a 1997 season as "next year" would understate heat and overstate frost. So we check whether there is a warming trend in annual mean temperature, and how sure we are.

**The bootstrap, in plain words:** we have 30 (year, temperature) pairs. Draw 30 pairs *at random, with replacement* (some years appear twice, some not at all), fit a line, and note its slope. Do that 2,000 times. The middle 90% of those 2,000 slopes is the **90% confidence interval**. It shows how much the slope could change with a slightly different set of years. It doesn't assume the errors follow a bell curve, which is why data scientists use it so much.

**Decide the rule BEFORE seeing the result.** Here's the rule: *if the 90% interval is entirely above 0, we shift past years' temperatures to 2026 levels using the trend. Otherwise we use no shift.* Fixing the rule first stops us from choosing whichever answer looks nicer. This is called **pre-registration**, and the same discipline in A/B testing prevents "p-hacking".

*(One honest limitation: this simple bootstrap treats years as independent. A warm year often follows a warm year, so the real interval is a bit wider. A "block bootstrap" handles that, and it comes up again in the bootcamp's statistics weeks.)*


In [ ]:
annual = w2.loc["1996":"2025", "T2M"].groupby(w2.loc["1996":"2025"].index.year).mean()
trend = fs.warming_trend(annual)
USE_TREND = trend["low"] > 0
TEMP_TREND = trend["slope_per_year"] if USE_TREND else 0.0
print(f"Trend: {10*trend['slope_per_year']:+.2f} degC per decade "
      f"(90% CI {10*trend['low']:+.2f} to {10*trend['high']:+.2f})")
print("Rule: shift to 2026 levels only if the whole interval is above 0 ->", "SHIFT" if USE_TREND else "NO SHIFT")
print(f"Example: a 1996 season is shifted by {TEMP_TREND*(2026-1996):+.2f} degC; a 2020 season by {TEMP_TREND*6:+.2f} degC")

ax = annual.plot(marker="o", color="#333", figsize=(8, 3))
x = annual.index.values
ax.plot(x, annual.mean() + trend["slope_per_year"] * (x - x.mean()), color="#c0392b", label="trend")
ax.set_ylabel("annual mean (degC)"); ax.legend(frameon=False); plt.tight_layout(); plt.show()

## Step 8: Possible next seasons, and how many we need

**How one possible future is made:**
1. Pick one of the 30 past years at random, **as a whole year**.
2. Shift its temperatures to 2026 warmth (Step 7's trend, if we're using it).
3. Multiply its rain by a random factor around 1 (spread ±10%, the uncertainty we chose in Lesson 2).
4. Add a small random temperature error (spread = `temp_sd` from Step 5).
5. Compute all the indicators for this future.

**Why whole years?** Weather comes in runs: a dry spell is *many* dry days in a row, and a hot July often comes with little rain. If we sampled single days at random, every future would be an average-looking mix with no droughts or cold snaps. That would hide exactly the risks a farmer cares about. Sampling whole years is a **block bootstrap** with one-year blocks.

**How many futures?** Each risk count is itself random. Run it again with a different seed and "23 in 100" might become "19 in 100". That wobble is called **Monte Carlo error**, and its size is roughly √(p(1−p)/n):

| n futures | wobble when the true rate is 20% |
|---|---|
| 100 | about ±4 in 100 |
| 1,000 | about ±1.3 in 100 |

The cell below shows the wobble with real numbers: five runs of 100, each with a different seed.

The fix is to **generate 1,000 futures and report the result per 100**. People still read "in 100", but the number is stable.


In [ ]:
RAIN_SD = config["precip_correction"].get("rain_uncertainty_fraction", 0.10)
demo_runs = {seed: fs.risk_counts(fs.make_futures(w2.loc["1996":"2025"], n=100, temp_trend=TEMP_TREND,
                                                  rain_sd=RAIN_SD, temp_sd=TEMP_SD, seed=seed))
             for seed in range(5)}
wobble = pd.DataFrame({s: {k: v["count"] for k, v in r.items()} for s, r in demo_runs.items()})
wobble.columns = [f"seed {s}" for s in wobble.columns]
print("Same method, five runs of 100 futures (count per 100):")
print(wobble.to_string())

futures = fs.make_futures(w2.loc["1996":"2025"], n=1000, temp_trend=TEMP_TREND,
                          rain_sd=RAIN_SD, temp_sd=TEMP_SD, seed=2026)
futures.to_csv(os.path.join(CACHE_DIR, "climate_futures_elora_2026.csv"))
print(f"\nFinal: {len(futures)} futures built from {futures.source_year.nunique()} different past years; "
      f"saved climate_futures_elora_2026.csv")

## Step 9: The odds, shown so a farmer can read them

`climate_odds` reports each risk two ways:
- **"in 100 futures"**: the forecast-style number
- **"past years: X of 30"**: what actually happened

Putting them side by side keeps us honest. The futures are built from those same 30 years, so they can't be much wiser than the history. They're mainly the history, adjusted to today's warmth, plus our uncertainty about the data.

**Product design: why "23 in 100" and squares?** Research on risk communication (in medicine, for example) shows people understand *counts of things* ("23 out of 100 people") more accurately than percentages or probabilities, and an **icon array** (100 squares, some coloured) makes it visual. That's why weather apps say "30% chance of rain" and still confuse people, and why we won't.

**Evidence for each risk = the weakest link.** The odds use rain (validated), temperature (validated if you corrected it) and the thresholds (assumed). The weakest of these is **assumed**, so that's the honest label for the odds. The thresholds are rules of thumb. If you can, check them against OMAFRA's field crop guides; once a threshold has a source, its label can go up.


In [ ]:
from fieldsense import viz
from fieldsense.profile import climate_odds

odds_evidence = ["power_precip_corrected", "power_temperature", "assumed_risk_thresholds"]
odds = climate_odds(futures, hist, evidence=odds_evidence, target_year=2026)
for k, r in odds["risks"].items():
    print(f"{round(100*r['share']):3d} in 100   (past: {r['past_years']:>8s})   {r['text']}")
print("\nWeakest evidence behind these odds:", reg.weakest(odds_evidence).upper())

fig = viz.plot_odds_icons(odds)
fig.savefig(os.path.join(DEMO_DIR, "climate_odds_elora_2026.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(DEMO_DIR, "climate_odds_elora_2026.svg"), bbox_inches="tight")
plt.show()

## Step 10: Update the field profiles to schema 0.2

We add `climate_odds` and the new indicators to each plot's profile, and set `"schema": "0.2"`.

**Backward-compatible change:** we only *added* things. Nothing was renamed or removed, so anything written to read 0.1 (like the Lesson 5 code) still works. The rule of thumb that many teams use (semantic versioning):

| Version step | What changed | Does old code still work? |
|---|---|---|
| **0.1 → 0.2** (minor) | Something was added | Yes |
| **1.x → 2.0** (major) | Something was renamed or removed | No: readers must update |

We also update the **gaps** list: if temperature is now validated, the gap "temperature has not been checked" is closed, so we remove it. A gap list that never shrinks means nobody is closing gaps.


In [ ]:
from fieldsense.profile import load_profile, save_profile, check_profile, climate_summary, SCHEMA_VERSION

climate_v2 = climate_summary(w2, years=(1996, 2025), rain_evidence="power_precip_corrected",
                             temp_evidence="power_temperature")
updated = []
for fname in sorted(f for f in os.listdir(PROFILE_DIR) if f.startswith("plot_") and f.endswith(".json")):
    path = os.path.join(PROFILE_DIR, fname)
    p = load_profile(path)
    p["schema"] = SCHEMA_VERSION
    p["climate"] = climate_v2
    p["climate_odds"] = odds
    if USE_TEMP_CORRECTION:
        p["gaps"] = [g for g in p["gaps"] if not g.startswith("Temperature has not")]
    problems = check_profile(p)
    save_profile(p, path)
    updated.append(fname)
    print(f"{fname}: schema {p['schema']}, {'OK' if not problems else problems}")

## Summary

Run this and paste the output back to me.


In [ ]:
print("=== Lesson 6 summary ===")
print("fieldsense", fs.__version__); sh(f"git -C {REPO_DIR} log --oneline -1")
for k, c in checks.items():
    print(f"temp {k:4s}: bias {c['bias_raw']:+.2f} -> {c['bias_corrected']:+.2f} degC, MAE {c['mae_raw']:.2f} -> {c['mae_corrected']:.2f}")
print(f"frost-free days error (test years): {FF_ERR_RAW:+.0f} -> {FF_ERR_COR:+.0f} days;  correction used: {USE_TEMP_CORRECTION}")
print(f"station median frost-free days: {fd_station.frost_free_days.median():.0f};  "
      f"corrected POWER 1996-2025 median: {hist.frost_free_days.median():.0f}")
print(f"warming trend: {10*trend['slope_per_year']:+.2f} degC/decade (90% CI {10*trend['low']:+.2f} to {10*trend['high']:+.2f}); shift used: {USE_TREND}")
print(f"temp_sd: {TEMP_SD:.2f} degC; futures: {len(futures)} from {futures.source_year.nunique()} years")
print("Monte Carlo wobble (min-max over 5 runs of 100):",
      {k: int(wobble.loc[k].max() - wobble.loc[k].min()) for k in wobble.index})
print("\nOdds for 2026:")
for k, r in odds["risks"].items():
    print(f"  {round(100*r['share']):3d} in 100  (past {r['past_years']})  {k}")
print("\nEvidence:", {e.id: e.label for e in reg.items.values()})
print("Profiles updated:", updated)

## What you learned

**Statistics**
- Error = **bias + noise**. A correction removes bias; only better data reduces noise.
- Use an **additive** correction for quantities with an arbitrary zero (temperature) and a **multiplicative** one for quantities that scale (rain).
- The **bootstrap** gives confidence intervals without assuming a bell curve.
- **Pre-registration:** write the decision rule before looking at the result.
- **Monte Carlo error** shrinks like 1/√n. Simulate more, report simply.

**Software and ML engineering**
- Branch → pull request → review → merge.
- **Data versioning:** save new versions (`weather_elora_v2.csv`) instead of overwriting.
- **Backward-compatible schema changes**, and semantic versioning (0.1 → 0.2).
- **One source of truth** for each definition (`year_indicators`), so two copies can't drift apart.

**Product**
- Show risk as **"X in 100"** with an icon array, next to what happened in real years.
- Label the odds by their **weakest** evidence, honestly.

---

## Interview practice (answer out loud first, then open the model answer)

**Q1. "Your model has low bias on the test set but a large mean absolute error. What does that tell you, and what would you do?"**
<details><summary>Model answer</summary>

Low bias with high MAE means errors are large but cancel out on average: the problem is *variance/noise*, not a systematic lean. A constant correction won't help. Options: better inputs (higher-resolution data, local stations), models that use more context (e.g. regress on wind/cloud for night temperatures), or accept and *quantify* the noise and propagate it as uncertainty. For FieldSense, frost **dates** depend mostly on bias, so the correction was the right tool for that decision even though daily MAE barely moved.
</details>

**Q2. "How would you put a confidence interval on a regression slope without assuming normal errors?"**
<details><summary>Model answer</summary>

Bootstrap: resample (x, y) pairs with replacement, refit, repeat ~1,000–10,000 times, take the 5th and 95th percentiles of the slopes for a 90% interval. Caveat: if observations are correlated in time (they are, for yearly climate), use a block bootstrap (resample consecutive blocks) or the interval will be too narrow.
</details>

**Q3. "You run a simulation and your risk estimate changes from 19% to 26% between runs. Is something wrong?"**
<details><summary>Model answer</summary>

Not necessarily. That's Monte Carlo error. With n = 100 and p ≈ 0.2, the standard error is √(0.2·0.8/100) ≈ 0.04, so ±4–8 points is normal. Fix by increasing n (1,000 gives ≈ ±1.3), fixing seeds for reproducibility, and reporting the estimate with its uncertainty or rounded appropriately.
</details>

**Q4. "You need to add a field to an API response other teams consume. How do you do it safely?"**
<details><summary>Model answer</summary>

Make it additive and optional (don't rename or remove existing fields), bump the minor version, document it, and keep a test that old payloads still validate. For a breaking change: bump the major version, run both versions in parallel for a migration period, and tell the consumers ahead of time.
</details>

**Next, Lesson 7: the rotation engine.** We compare your current rotation with alternatives, using the barcode, the living cover and today's odds, and every recommendation will carry its weakest-link evidence label.
